# Session 6 — Real case: Electronics replenishment exceptions

**Salih Tutun, PhD** · DAT 5668 · Module 3 · Explain → practice together

## How to use this notebook

1. Read the short **Topic** cells (Part I). Do not scroll into practice yet.  
2. In Part II, every stop is: **explain markdown → small code cell → run → talk**.  
3. Cues:

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Code is ready. Read comments. Run. Do not edit. |
| **TRY THIS** | You type `<<< FILL`. Re-run. Check. |
| **CHANGE ME** | Change the marked value. Re-run. **PAIR 20s** discuss. |
| **CHECK** | Print PASS / NOT YET |

Partner rule: both can say who **releases the daily order** and who **stamps the promo override**.

![Two families at one desk](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)
![Hybrid design](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

*Images load from GitHub. If blank: hard-refresh or reopen.*


## Practice together tonight (change → see → discuss)

For **CHANGE ME** cells:
1. Change only the marked line  
2. Re-run that cell  
3. **PAIR 20s** — what changed?  
4. Two voices share — then we continue  

We are not hunting a secret answer. We are learning what the real desk does when YOU change a rule.


# Part I — Topic (explain before coding)

## The real desk

You own **Electronics** replenishment. Real log: `inventory_dataset_full.csv` (730 Electronics days).

| Column | Meaning |
| --- | --- |
| `Current_Inventory` | Stock you can measure (state) |
| `Demand`, `Stockout_Units` | What customers wanted / missed |
| `Action_Taken` | `Order` / `Maintain` / `Reduce` |
| `Profit`, `Holding_Cost`, `Stockout_Cost` | Money (reward language) |
| `Is_Promotional` | Campaign day — needs a document path |

![Same inventory loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-loop.png)

```text
observe inventory → decide action → demand → profit/costs → next inventory
```

![Numeric path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)
![Document path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Human stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)

---

## The go / no-go memo (tonight's decision artifact)

A **go / no-go memo** is not a vibes vote and not a ChatGPT essay.  
It is the **one-page design decision** you would hand a desk lead / CFO:

> Do we **GO** (automate this process with a named family), or **NO-GO** (do not automate yet)?  
> If we GO — **who releases what**, on which path, with which handoff?

![Hybrid design](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

### What GO means

**GO** = we authorize automation **under the memo's rules**.

| GO choice | Meaning on THIS desk |
| --- | --- |
| **RL** | A numeric policy may **release** the daily Reduce / Maintain / Order from `Current_Inventory` + reward language |
| **LLM** | A language agent may draft from documents — still usually needs a **human stamp** before money moves |
| **hybrid** | **Both paths are true**: RL/basestock owns the daily number; LLM only drafts promo/supplier overrides for a person |

### What NO-GO means

**NO-GO** = **do not automate** (yet).

Use **do not automate** when any of these is true:
- you cannot name a **reward** the CFO would sign (margin / sales / profit language missing)
- one bad exploration order is **unacceptable** while you learn
- nobody can name who is **accountable** if the override is wrong
- the “state” is still a fuzzy email with no measurable stock path

NO-GO is a **professional answer**, not a failure. Weak answer: “use AI for everything.”

### The four recommendation words (type exactly)

| Word | When it fits |
| --- | --- |
| `RL` | Repeated numeric state + named reward; little/no document exception |
| `LLM` | Work is mostly documents; inventory number is not the main decision |
| `hybrid` | Numeric daily order **and** document exceptions (tonight’s Electronics log) |
| `do not automate` | Reward unsigned, unsafe to explore, or release owner missing |

### Strong memo vs weak memo

| Strong (PASS) | Weak (I send it back) |
| --- | --- |
| Names **MDP**: states, actions `{0,1,2}`, reward with profit/sales/margin, why repeatable | “AI will optimize inventory” |
| Names **LLM**: goal, tools, knowledge, **human_handoff** | “ChatGPT reads emails” |
| Says who **releases the daily order** vs who **stamps the promo override** | One vague “owner: the AI” |
| `because` cites **this** Electronics evidence (≥ 40 chars) | No process, no data, no checkpoint |
| Recommendation fits the evidence (usually **hybrid** tonight) | Pure `LLM` for a numeric stock order |

### Teach aloud (write these three lines)

1. **Prediction ≠ agent** — a forecast is a tool; it does not release the order.  
2. **Proposal ≠ release** — the LLM may draft; a person (or RL policy) releases.  
3. **Stock number ≠ promo brief** — numbers → MDP/RL path; documents → LLM + stamp.

**Tonight’s evidence path:** load real rows → define MDP + LLM sides → vote → assemble `MEMO` → `review_memo` must print **PASS**.

**JUMP BACK** — do not open Stop 0 until I say OPEN.


---
# Part II — Practice (small cells: explain → run)

We split code on purpose: one idea per cell so every line is readable.


## Stop 0 — Find the dataset

**Goal:** attach Drive (Colab) or confirm local files exist.

**What the next cell does**
1. Import `Path` (clean file paths)  
2. Name the Colab Drive folder  
3. Try to mount Google Drive  
4. List CSV files and check for `inventory_dataset_full.csv`


In [ ]:
# STOP 0 — RUN WITH ME
# Find inventory_dataset_full.csv (Drive in Colab, or local).

from pathlib import Path  # clean path joins

DRIVE_ROOT = Path("/content/drive/MyDrive/DAT5668/data")  # Colab shared data folder

try:
    from google.colab import drive  # type: ignore  # Colab-only import
    drive.mount("/content/drive", force_remount=False)  # attach Drive once
except ImportError:
    pass  # laptop / Cursor: skip mount

# Optional: confirm the Session 6 CSV is visible when Drive exists
if DRIVE_ROOT.exists():
    ok = (DRIVE_ROOT / "inventory_dataset_full.csv").exists()
    print("inventory_dataset_full.csv:", "YES" if ok else "MISSING")


## Stop 1 — Load the real Electronics desk

![Numeric desk](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

We load **only** `Product == "Electronics"` rows, then print real facts.


### 1a — Imports (RUN WITH ME)

**What this cell does:** bring in tools we need for files, counting, CSV, averages, and charts.


In [ ]:
# STOP 1a-1 — RUN WITH ME · imports

from pathlib import Path          # file paths
from collections import Counter   # counts for actions / demand bins
import csv                        # read CSV rows as dicts
import statistics                 # mean() for summary stats

try:
    import matplotlib.pyplot as plt  # optional charts
except ImportError:
    plt = None  # numbers still work without plots


### 1a — Settings (RUN WITH ME)

**What this cell does:** name the Drive folder and the product line we study tonight.


In [ ]:
# STOP 1a-2 — RUN WITH ME · settings

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"  # Colab data root
PRODUCT = "Electronics"  # product line we study tonight


### 1a — Helper: `course_data` (RUN WITH ME)

**What this cell does:** define a function that finds a CSV on Drive first, then in local course folders.


In [ ]:
# STOP 1a-3 — RUN WITH ME · define course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local course folders
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)  # Colab path first
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("../../Contents/Module 4"),
        Path("../Contents/Module 4"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)  # local fallbacks
    for path in candidates:
        if path.exists():
            return path  # first hit wins
    raise FileNotFoundError("Could not find " + name)


### 1a — Helper: `_flag` (RUN WITH ME)

**What this cell does:** turn CSV flags like `"1"` / `"True"` into real Python `True`/`False`.


In [ ]:
# STOP 1a-4 — RUN WITH ME · define _flag

def _flag(val) -> bool:
    # CSV stores booleans as strings like "1" / "True"
    return str(val).strip() in {"1", "True", "true", "YES", "Yes"}


### 1a — Helper: `load_electronics_desk` (RUN WITH ME)

**What this cell does:** open the CSV, keep Electronics rows, convert types, sort by date.


In [ ]:
# STOP 1a-5 — RUN WITH ME · define load_electronics_desk

def load_electronics_desk():
    rows = []  # one dict per Electronics day
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):  # {column: value}
            if (row.get("Product") or "").strip() != PRODUCT:
                continue  # skip other product lines
            rows.append(
                {
                    "Date": row["Date"],
                    "Current_Inventory": int(float(row["Current_Inventory"])),  # state
                    "Demand": int(float(row["Demand"])),
                    "Units_Sold": int(float(row["Units_Sold"])),
                    "Stockout_Units": int(float(row["Stockout_Units"])),
                    "Lead_Time": int(float(row["Lead_Time"])),
                    "Price": float(row["Price"]),
                    "Is_Promotional": _flag(row.get("Is_Promotional", 0)),  # exception flag
                    "Is_Weekend": _flag(row.get("Is_Weekend", 0)),
                    "Action_Taken": (row.get("Action_Taken") or "").strip(),  # Order/Maintain/Reduce
                    "Order_Quantity": int(float(row.get("Order_Quantity") or 0)),
                    "Order_Placed": _flag(row.get("Order_Placed", 0)),
                    "Holding_Cost": float(row.get("Holding_Cost") or 0),
                    "Stockout_Cost": float(row.get("Stockout_Cost") or 0),
                    "Profit": float(row.get("Profit") or 0),  # reward language
                    "Revenue": float(row.get("Revenue") or 0),
                    "Season": (row.get("Season") or "").strip(),
                    "Reorder_Point": int(float(row.get("Reorder_Point") or 0)),
                }
            )
    if not rows:
        raise ValueError("No Electronics rows found")
    rows.sort(key=lambda r: r["Date"])  # chronological
    return rows


### 1a — Run the load (RUN WITH ME)

**What this cell does:** call the loader once and keep `ROWS` for the rest of class.


In [ ]:
# STOP 1a-6 — RUN WITH ME · load once

ROWS = load_electronics_desk()  # full Electronics log
N_DAYS = len(ROWS)              # e.g. 730
print("Electronics days:", N_DAYS)  # one confirmation line


### 1a — Real summary numbers (RUN WITH ME)

**What this cell does:** compute averages and counts students will quote in the memo.


In [ ]:
# STOP 1a-7 — RUN WITH ME · summary stats

AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)  # for Low/Med/High cuts
AVG_INV = statistics.mean(r["Current_Inventory"] for r in ROWS)
PROMO_DAYS = [r for r in ROWS if r["Is_Promotional"]]  # document-path days
STOCKOUT_DAYS = [r for r in ROWS if r["Stockout_Units"] > 0]
ACTIONS = Counter(r["Action_Taken"] for r in ROWS)  # historical actions

# Short desk card (keep output small)
print(
    "avg inv/demand:", round(AVG_INV, 1), "/", round(AVG_DEMAND, 1),
    "| promo:", len(PROMO_DAYS),
    "| stockout days:", len(STOCKOUT_DAYS),
    "| actions:", dict(ACTIONS),
)


### 1a — Demand bins + charts (RUN WITH ME)

**What this cell does:** bin real daily demand into $d \in \{0,1,2\}$ (readable MDP style) and plot.


In [ ]:
# STOP 1a-8 — RUN WITH ME · demand bins (+ optional chart)

demands = sorted(r["Demand"] for r in ROWS)
q1, q2 = demands[len(demands) // 3], demands[(2 * len(demands)) // 3]  # third cuts


def demand_bin(d: int) -> int:
    if d <= q1:
        return 0  # low demand bin
    if d <= q2:
        return 1  # mid
    return 2  # high


_counts = Counter(demand_bin(r["Demand"]) for r in ROWS)
DEMAND_P = {k: _counts[k] / N_DAYS for k in (0, 1, 2)}  # probabilities
print("P(d):", {k: round(v, 3) for k, v in DEMAND_P.items()})

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.bar(["0 low", "1 mid", "2 high"], [DEMAND_P[0], DEMAND_P[1], DEMAND_P[2]], color="#2F5D50")
    ax.set_ylabel("P(d)")
    ax.set_title("MDP demand bins (Electronics)")
    ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()


### 1a — Desk dashboard (RUN WITH ME)

**What this cell does:** plot inventory + profit on a sample of real days, mark promo days. Validation that both numeric and document paths exist in the same log.


In [ ]:
# STOP 1a-9 — RUN WITH ME · desk dashboard (plots + validation)

SAMPLE = ROWS[:: max(1, N_DAYS // 120)]  # ~120 points so the chart stays readable
xs = list(range(len(SAMPLE)))
inv = [r["Current_Inventory"] for r in SAMPLE]
profit = [r["Profit"] for r in SAMPLE]
promo_x = [i for i, r in enumerate(SAMPLE) if r["Is_Promotional"]]

# Validation numbers (quote these in the memo later)
n_promo = len(PROMO_DAYS)
n_stockout = len(STOCKOUT_DAYS)
mean_profit = statistics.mean(r["Profit"] for r in ROWS)
print(
    "validate:",
    "days", N_DAYS,
    "| promo", n_promo,
    "| stockout days", n_stockout,
    "| mean Profit", round(mean_profit, 1),
)
assert N_DAYS > 0 and n_promo > 0 and n_stockout >= 0
print("CHECK desk — PASS (numeric path + promo days both present)")

if plt is not None:
    fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True)
    axes[0].plot(xs, inv, color="#2F5D50", lw=1.2)
    axes[0].set_ylabel("Inventory")
    axes[0].set_title("Electronics desk sample (every ~N/120 day)")
    for x in promo_x:
        axes[0].axvline(x, color="#C45C26", alpha=0.25, lw=1)
    axes[1].plot(xs, profit, color="#1F4E79", lw=1.0)
    axes[1].set_ylabel("Profit")
    axes[1].set_xlabel("sample index")
    for x in promo_x:
        axes[1].axvline(x, color="#C45C26", alpha=0.25, lw=1)
    axes[1].legend(
        handles=[
            plt.Line2D([0], [0], color="#C45C26", alpha=0.6, label="promo day"),
        ],
        loc="upper right",
        fontsize=8,
    )
    fig.tight_layout()
    plt.show()


### 1b — TRY THIS (read the printout)

Fill three facts from Stop 1a. No inventing numbers.


In [ ]:
# STOP 1b — TRY THIS

n_days_guess = None        # <<< FILL: Days in log (integer)
promo_count_guess = None   # <<< FILL: Promo days (integer)
has_profit_column = None   # <<< FILL: True/False — does the log include Profit?

print("Your answers:", n_days_guess, promo_count_guess, has_profit_column)

if None in (n_days_guess, promo_count_guess, has_profit_column):
    print("NOT YET — fill all three from the Stop 1a printout")
else:
    ok = (
        n_days_guess == N_DAYS
        and promo_count_guess == len(PROMO_DAYS)
        and has_profit_column is True
    )
    print("CHECK 1 —", "PASS" if ok else "NOT YET — compare to Days in log / Promo days")


## Stop 2 — Map real columns → state and action

![MDP mapping](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

| Data `Action_Taken` | Code $a$ |
| --- | --- |
| Reduce | 0 |
| Maintain | 1 |
| Order | 2 |

State rule vs average demand:
- **Low** if inventory $< 0.75 \times$ avg demand  
- **Medium** if $< 1.5 \times$ avg demand  
- **High** otherwise  

Basestock practice policy: Low→Order(2), Medium→Maintain(1), High→Reduce(0).


### 2a — Define `observe_state` (RUN WITH ME)


In [ ]:
# STOP 2a-1 — RUN WITH ME · observe_state

def observe_state(inventory, avg_demand=AVG_DEMAND):
    # Map raw inventory → Low / Medium / High vs average demand
    inv = float(inventory)
    if inv < 0.75 * avg_demand:
        return "Low"
    if inv < 1.5 * avg_demand:
        return "Medium"
    return "High"


### 2a — Define action helpers (RUN WITH ME)


In [ ]:
# STOP 2a-2 — RUN WITH ME · action helpers

def action_to_code(action_taken: str) -> int:
    # Historical text → class codes {0,1,2}
    return {"Reduce": 0, "Maintain": 1, "Order": 2}.get(action_taken, -1)


def decide_basestock(state: str) -> int:
    # Practice policy: Low→Order, Medium→Maintain, High→Reduce
    return {"Low": 2, "Medium": 1, "High": 0}[state]


### 2a — Show five real days (RUN WITH ME)


In [ ]:
# STOP 2a-3 — RUN WITH ME · peek 3 real days

for r in ROWS[0:3]:
    s = observe_state(r["Current_Inventory"])  # our state
    # date | inv | state | historical action code | basestock suggestion
    print(r["Date"], r["Current_Inventory"], s, action_to_code(r["Action_Taken"]), decide_basestock(s))


### 2b — TRY THIS — classify three inventories


In [ ]:
# STOP 2b — TRY THIS

inv_A = ROWS[0]["Current_Inventory"]  # given from the real log
inv_B = int(AVG_DEMAND)               # near average
inv_C = int(2.5 * AVG_DEMAND)         # clearly high

state_A = ""  # <<< FILL: "Low" / "Medium" / "High"
state_B = ""  # <<< FILL
state_C = ""  # <<< FILL
a_A = None    # <<< FILL: 0 / 1 / 2 from decide_basestock
a_B = None    # <<< FILL
a_C = None    # <<< FILL

print(inv_A, state_A, a_A)
print(inv_B, state_B, a_B)
print(inv_C, state_C, a_C)

if not state_A or not state_B or not state_C or None in (a_A, a_B, a_C):
    print("NOT YET — fill states and actions")
else:
    ok = (
        (state_A, a_A) == (observe_state(inv_A), decide_basestock(observe_state(inv_A)))
        and (state_B, a_B) == (observe_state(inv_B), decide_basestock(observe_state(inv_B)))
        and (state_C, a_C) == (observe_state(inv_C), decide_basestock(observe_state(inv_C)))
    )
    print("CHECK 2 —", "PASS" if ok else "NOT YET — use observe_state / decide_basestock")


### 2c — CHANGE ME · discuss thresholds together

**PAIR 20s.** Change the cutoffs. Re-run. Watch how many days become Low / Medium / High.

**Discuss:** If almost everything is “Low,” your basestock will Order too often. What does that do to holding cost?


In [ ]:
# STOP 2c — CHANGE ME · then discuss

LOW_CUT = 0.75   # <<< CHANGE ME: inventory < LOW_CUT * avg → Low
HIGH_CUT = 1.5   # <<< CHANGE ME: else Medium until HIGH_CUT * avg


def observe_state_custom(inventory, avg_demand=AVG_DEMAND, low_cut=LOW_CUT, high_cut=HIGH_CUT):
    inv = float(inventory)
    if inv < low_cut * avg_demand:
        return "Low"
    if inv < high_cut * avg_demand:
        return "Medium"
    return "High"


counts = {"Low": 0, "Medium": 0, "High": 0}
for r in ROWS:
    counts[observe_state_custom(r["Current_Inventory"])] += 1  # recount whole log

print(counts)  # discuss: too many Low → over-ordering?
print("share:", {k: round(v / N_DAYS, 3) for k, v in counts.items()})

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    labels = ["Low", "Medium", "High"]
    ax.bar(labels, [counts[k] for k in labels], color=["#C45C26", "#2F5D50", "#1F4E79"])
    ax.set_ylabel("days")
    ax.set_title(f"MDP state mix (cuts {LOW_CUT} / {HIGH_CUT})")
    fig.tight_layout()
    plt.show()


### 2d — CHANGE ME · rewrite the policy and compare to history

**PAIR 20s.** Change what Low/Medium/High should do. Re-run.

**Discuss:** On how many real days would YOUR policy match the log’s `Action_Taken`?


In [ ]:
# STOP 2d — CHANGE ME · policy vs history

POLICY = {
    "Low": 2,     # <<< CHANGE ME: 0 Reduce / 1 Maintain / 2 Order
    "Medium": 1,  # <<< CHANGE ME
    "High": 0,    # <<< CHANGE ME
}

agree = 0
# rows = true Action_Taken code, cols = our POLICY action
matrix = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
for r in ROWS:
    s = observe_state(r["Current_Inventory"])
    true_a = action_to_code(r["Action_Taken"])
    pol_a = POLICY[s]
    matrix[true_a][pol_a] += 1
    if pol_a == true_a:
        agree += 1  # your policy matched the log that day

rate = agree / N_DAYS
print(agree, "/", N_DAYS, "agreement", round(rate, 3))  # discuss: disagreement needs a story
print("CHECK policy —", "PASS" if 0.2 <= rate <= 0.95 else "LOOK — extreme agreement/disagreement; discuss why")

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.2))
    axes[0].bar(["agree", "disagree"], [agree, N_DAYS - agree], color=["#2F5D50", "#8B3A2A"])
    axes[0].set_title("Policy vs history")
    axes[0].set_ylabel("days")
    im = axes[1].imshow(matrix, cmap="Greens")
    axes[1].set_xticks([0, 1, 2], ["pol 0", "pol 1", "pol 2"])
    axes[1].set_yticks([0, 1, 2], ["true 0", "true 1", "true 2"])
    axes[1].set_title("Confusion (true Action_Taken × POLICY)")
    for i in range(3):
        for j in range(3):
            axes[1].text(j, i, str(matrix[i][j]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    fig.tight_layout()
    plt.show()


## Stop 3 — Solve one REAL day

![Real day](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)
![Promo path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/llm-tool-loop.png)

We pick a real non-promo day and a real promo day from `ROWS`, then build a promo brief from that date's numbers.


### 3a — Helper to pick a real day (RUN WITH ME)


In [ ]:
# STOP 3a-1 — RUN WITH ME · pick_real_day

def pick_real_day(prefer_promo: bool):
    # First real day matching promo / not-promo
    for r in ROWS:
        if r["Is_Promotional"] is prefer_promo:
            return r
    return ROWS[0]


### 3a — Load two real days (RUN WITH ME)


In [ ]:
# STOP 3a-2 — RUN WITH ME · two real days

DAY_NORMAL = pick_real_day(False)  # not promotional
DAY_PROMO = pick_real_day(True)    # promotional

# Compact print: date, inv, promo flag, historical action
print("normal:", DAY_NORMAL["Date"], "inv", DAY_NORMAL["Current_Inventory"], "promo", DAY_NORMAL["Is_Promotional"], DAY_NORMAL["Action_Taken"])
print("promo: ", DAY_PROMO["Date"], "inv", DAY_PROMO["Current_Inventory"], "promo", DAY_PROMO["Is_Promotional"], DAY_PROMO["Action_Taken"])


### 3a — Build the exception document (RUN WITH ME)

The brief uses **real numbers** from `DAY_PROMO`, but it is still a **document** (not a stock column).


In [ ]:
# STOP 3a-3 — RUN WITH ME · PROMO_BRIEF (document, not a stock column)

PROMO_BRIEF = {
    "date": DAY_PROMO["Date"],
    "subject": "Electronics campaign live — " + DAY_PROMO["Date"],
    # Body is grounded in real numbers from that day
    "body": (
        "Promo day demand=" + str(DAY_PROMO["Demand"])
        + ", inventory=" + str(DAY_PROMO["Current_Inventory"])
        + ", stockout=" + str(DAY_PROMO["Stockout_Units"])
        + ". Do not auto-change Order_Quantity without desk-lead stamp."
    ),
}
print(PROMO_BRIEF["subject"])  # one line is enough before discussion


### 3b — TRY THIS — promo day questions


In [ ]:
# STOP 3b — TRY THIS

promo_date = ""  # <<< FILL: DAY_PROMO["Date"]
base_a = None    # <<< FILL: basestock action 0/1/2 for that day's inventory
needs_override_draft = None  # <<< FILL: True/False
email_inside_inventory_number = None  # <<< FILL: True/False

print(promo_date, base_a, needs_override_draft, email_inside_inventory_number)

if not promo_date or base_a is None or needs_override_draft is None or email_inside_inventory_number is None:
    print("NOT YET")
else:
    ok = (
        promo_date == DAY_PROMO["Date"]
        and base_a == decide_basestock(observe_state(DAY_PROMO["Current_Inventory"]))
        and needs_override_draft is True
        and email_inside_inventory_number is False
    )
    print("CHECK 3 —", "PASS" if ok else "NOT YET")


### 3c — CHANGE ME · pick another real day and discuss

**PAIR 20s.** Set `DAY_INDEX` to another row (0 … N_DAYS-1). Re-run.

**Discuss:** Is this day promotional? Would you draft an override? Who stamps it?


In [ ]:
# STOP 3c — CHANGE ME · another real day

DAY_INDEX = 100  # <<< CHANGE ME: 0 … N_DAYS-1
day = ROWS[DAY_INDEX]
state = observe_state(day["Current_Inventory"])

# Compact facts for discussion
print(day["Date"], "inv", day["Current_Inventory"], state, "promo", day["Is_Promotional"], "hist", day["Action_Taken"])
would_draft_override = day["Is_Promotional"]  # <<< CHANGE ME if you disagree
print("draft override?", would_draft_override)


## Stop 4 — Choose the family (TRY THIS)

![Choose family](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/two-families.png)

Type exactly: `LLM`, `RL`, `hybrid`, or `do not automate`.


In [ ]:
# STOP 4 — TRY THIS

# A) Only Current_Inventory → Order/Maintain/Reduce daily; profit costs; no briefs
case_A = ""  # <<< FILL

# B) One campaign brief; human stamp; not the daily controller alone
case_B = ""  # <<< FILL

# C) THIS desk: 730 inventory days AND many Is_Promotional days
case_C = ""  # <<< FILL

# D) No signed reward policy; autonomous orders forbidden for now
case_D = ""  # <<< FILL

print(case_A, case_B, case_C, case_D)

if any(not x.strip() for x in [case_A, case_B, case_C, case_D]):
    print("NOT YET — fill A–D")
else:
    ok = [case_A, case_B, case_C, case_D] == ["RL", "LLM", "hybrid", "do not automate"]
    print("CHECK 4 —", "PASS" if ok else "NOT YET — expected RL / LLM / hybrid / do not automate")


## Stop 5 — DEFINE the MDP side (TRY THIS)

![MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

This is the **numeric half** of tonight's go/no-go memo. Use **real column names** (`Current_Inventory`, `Profit`, …).

| MDP piece | Fill with |
| --- | --- |
| **States** | inventory band from `Current_Inventory` |
| **Actions** | `[0, 1, 2]` = Reduce / Maintain / Order |
| **Reward** | Profit / sales / margin language |
| **Why repeatable** | daily Electronics over `N_DAYS` |

After you FILL, the next cell draws your MDP card and validates the shape.


In [ ]:
# STOP 5 — TRY THIS

mdp_states = ""          # <<< FILL: mention Current_Inventory
mdp_actions = []         # <<< FILL: [0, 1, 2]
mdp_reward = ""          # <<< FILL: must mention profit OR sales OR margin
mdp_why_repeatable = ""  # <<< FILL: daily / Electronics / N_DAYS

print("states:", mdp_states or "(FILL)")
print("actions:", mdp_actions or "(FILL)")
print("reward:", mdp_reward or "(FILL)")
print("why:", mdp_why_repeatable or "(FILL)")
print("Evidence N_DAYS =", N_DAYS)

if not mdp_states or not mdp_actions or not mdp_reward or not mdp_why_repeatable:
    print("NOT YET")
else:
    ok = (
        mdp_actions == [0, 1, 2]
        and any(w in mdp_reward.lower() for w in ("margin", "sales", "profit"))
        and "inventory" in mdp_states.lower()
    )
    print("CHECK 5 —", "PASS" if ok else "NOT YET")


### 5b — MDP card + validation (RUN WITH ME)

Re-run after you change Stop 5 fills. This is the visual you will defend in the memo.


In [ ]:
# STOP 5b — RUN WITH ME · MDP card (plot + validation)

_reward_words = ("margin", "sales", "profit")
mdp_ok = (
    bool((mdp_states or "").strip())
    and mdp_actions == [0, 1, 2]
    and any(w in (mdp_reward or "").lower() for w in _reward_words)
    and bool((mdp_why_repeatable or "").strip())
    and "inventory" in (mdp_states or "").lower()
)
print("MDP validation —", "PASS" if mdp_ok else "NOT YET — fix Stop 5 FILLs first")
print("states:", (mdp_states or "(empty)")[:80])
print("actions:", mdp_actions or "(empty)")
print("reward:", (mdp_reward or "(empty)")[:80])
print("why:", (mdp_why_repeatable or "(empty)")[:80])

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 4)
    ax.axis("off")
    ax.set_title("MDP half of the go / no-go memo", fontsize=12, pad=8)
    boxes = [
        (0.3, 1.2, 2.2, 2.0, "STATES", (mdp_states or "FILL")[:42], "#E8F0EC"),
        (2.8, 1.2, 2.2, 2.0, "ACTIONS", str(mdp_actions or "FILL"), "#E8EEF5"),
        (5.3, 1.2, 2.2, 2.0, "REWARD", (mdp_reward or "FILL")[:42], "#F7EDE6"),
        (7.8, 1.2, 1.9, 2.0, "REPEAT?", (mdp_why_repeatable or "FILL")[:36], "#F3F0E8"),
    ]
    for x, y, w, h, title, body, color in boxes:
        rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="#333333", lw=1.2)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h - 0.35, title, ha="center", va="top", fontsize=9, fontweight="bold")
        ax.text(x + w / 2, y + h / 2 - 0.1, body, ha="center", va="center", fontsize=7)
    status = "PASS" if mdp_ok else "NOT YET"
    ax.text(
        5,
        0.45,
        f"validation: {status}",
        ha="center",
        fontsize=10,
        color="#2F5D50" if mdp_ok else "#8B3A2A",
    )
    fig.tight_layout()
    plt.show()


## Stop 6 — DEFINE the LLM side on the real promo brief

![LLM](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)


### 6a — Re-read the real brief (RUN WITH ME)


In [ ]:
# STOP 6a — RUN WITH ME · re-read promo day

base = decide_basestock(observe_state(DAY_PROMO["Current_Inventory"]))  # numeric path
print(PROMO_BRIEF["subject"])
print("basestock a =", base, "| history =", DAY_PROMO["Action_Taken"], DAY_PROMO["Order_Quantity"])


### 6b — TRY THIS — LLM design + one draft override

Remember: **proposal ≠ release**.


In [ ]:
# STOP 6b — TRY THIS

llm_goal = ""            # <<< FILL
llm_tools = []           # <<< FILL: list of tool name strings
llm_knowledge = ""       # <<< FILL
llm_human_handoff = ""   # <<< FILL: who stamps?
proposed_override_a = None  # <<< FILL: 0, 1, or 2
proposal_reason = ""        # <<< FILL: must include the real promo DATE

print("goal:", llm_goal or "(FILL)")
print("tools:", llm_tools or "(FILL)")
print("knowledge:", llm_knowledge or "(FILL)")
print("handoff:", llm_human_handoff or "(FILL)")
print("proposal:", proposed_override_a, "|", proposal_reason or "(FILL)")

if (
    not llm_goal
    or not llm_tools
    or not llm_human_handoff
    or proposed_override_a is None
    or not proposal_reason
):
    print("NOT YET")
else:
    ok = proposed_override_a in (0, 1, 2) and DAY_PROMO["Date"] in proposal_reason
    print("CHECK 6 —", "PASS" if ok else "NOT YET — include the real DATE in proposal_reason")


### 6c — CHANGE ME · draft a different override and discuss

**PAIR 20s.** Change `proposed_override_a`. Re-run.

**Discuss:** On the real promo day, when is Order (2) vs Maintain (1) the better draft — and who can release it?


In [ ]:
# STOP 6c — CHANGE ME · override draft

base = decide_basestock(observe_state(DAY_PROMO["Current_Inventory"]))
proposed_override_a = 2  # <<< CHANGE ME: 0 / 1 / 2

print("promo", DAY_PROMO["Date"], "basestock", base, "your draft", proposed_override_a)
# discuss: if draft != basestock, what risk did the brief add?


## Stop 7 — Go / no-go memo for THIS real Electronics desk

![Memo](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

This stop is the **center of Session 6**. Everything before it was evidence.  
Now you write the **go / no-go memo** that `review_memo` can grade.

### Reminder — GO vs NO-GO on this desk

| Decision | What you are authorizing |
| --- | --- |
| **GO · RL** | Daily order quantity may auto-release from inventory state + reward |
| **GO · LLM** | Document agent may draft; release still needs a named handoff |
| **GO · hybrid** | RL/basestock releases daily order; LLM drafts promo override; **person stamps** |
| **NO-GO · do not automate** | Keep humans in the loop; do not ship an agent that spends money yet |

For `inventory_dataset_full.csv` Electronics (730 days **and** many `Is_Promotional` days), the honest GO is almost always **hybrid**.

### Memo anatomy (every field must be fillable)

| Field in `MEMO` | Comes from | Must answer |
| --- | --- | --- |
| `recommendation` | Stop 7a vote | Exactly one of: `LLM` / `RL` / `hybrid` / `do not automate` |
| `mdp_design` | Stop 5 | states · actions · reward · why_repeatable |
| `llm_design` | Stop 6 | goal · tools · knowledge · human_handoff |
| `releases_daily_order` | Stop 7b | Who **releases** Reduce/Maintain/Order each day? |
| `releases_email_override` | Stop 7b | Who **stamps** the promo/supplier override? |
| `because` | Stop 7b | ≥ 40 characters; mention Electronics or `inventory_dataset_full` |

### Class flow for Stop 7

1. **Vote** (7a) — say the recommendation out loud before typing the rest  
2. **Release owners + because** (7b) — accountability sentences  
3. **Define `review_memo`** (7c) — the grading checklist is the course standard  
4. **Assemble `MEMO`** — one dict; run check → **go / no-go board**

Partner rule still holds: both partners must be able to name **both** release owners.


### What “go / no-go” means in one minute (teach before you vote)

**Go / no-go** is a **ship decision** for an agentic process — not a model-accuracy score.

```text
evidence (real rows + promo brief)
        ↓
design both sides (MDP + LLM)
        ↓
recommendation = GO family  OR  NO-GO
        ↓
name who RELEASES each path
        ↓
review_memo → PASS / NOT YET
```

| If you choose… | You are saying… |
| --- | --- |
| **GO hybrid** | “Automate the **number** with RL/basestock; keep a **human** on document overrides.” |
| **GO RL only** | “Exceptions as documents are rare/irrelevant — I doubt that on this log.” |
| **GO LLM only** | “The daily stock order is not numeric enough” — usually **wrong** for Electronics inventory |
| **NO-GO** | “Do not let software release money/orders until reward + owners are clear.” |

![Two families](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)


### Worked hybrid memo (board pattern — same fields you will type)

Keep this table in mind while you FILL. Wording may differ; structure may not.

| Field | Worked answer for THIS Electronics desk |
| --- | --- |
| Recommendation | **hybrid** |
| MDP states | `Current_Inventory` bands (Low / Medium / High vs avg Demand) |
| MDP actions | `[0, 1, 2]` = Reduce / Maintain / Order |
| MDP reward | Profit language (sales/margin − holding − stockout) |
| MDP why repeatable | Every day across `N_DAYS` Electronics rows |
| LLM goal | Read promo/supplier briefs; propose override only when risk changes |
| LLM tools | e.g. `read_promo_brief`, `create_override_ticket`, lookup day in log |
| LLM human_handoff | Desk lead stamps before `Order_Quantity` changes |
| Releases daily order | **RL / basestock** on `Current_Inventory` (not the LLM) |
| Releases email override | **Desk lead (person)** after LLM drafts |
| Because | Cite Electronics + `inventory_dataset_full` + both paths |

**Weak patterns to avoid:** “use AI,” missing handoff, LLM releases the daily order, empty `because`.


In [ ]:
# STOP 7-PRE — RUN WITH ME · print the memo contract (read aloud)

print("GO / NO-GO MEMO CONTRACT")
print("-----------------------")
print("1. recommendation ∈ {LLM, RL, hybrid, do not automate}")
print("2. mdp_design: states + actions + reward(profit|sales|margin) + why_repeatable")
print("3. llm_design: goal + tools + knowledge + human_handoff")
print("4. releases_daily_order  = who RELEASES the numeric order")
print("5. releases_email_override = who STAMPS the document override")
print("6. because ≥ 40 chars and mentions Electronics or inventory_dataset_full")
print()
print("PASS means: complete enough to defend in front of a desk lead.")
print("NOT YET means: fix the listed gaps — do not ship.")


### 7a — Class vote (TRY THIS) — the GO / NO-GO recommendation

**Neighbor 20s. Two voices. Vote before anyone types the rest of the memo.**

Type exactly one word:
`LLM` · `RL` · `hybrid` · `do not automate`

Tonight’s evidence push: 730 inventory days **and** promo document days → expect **hybrid**.  
Challenge any pure-`LLM` vote for a numeric stock desk.


In [ ]:
# STOP 7a — TRY THIS

say_it = ""  # <<< FILL: LLM / RL / hybrid / do not automate
print("Vote:", say_it or "(FILL)")
print("Evidence — promo days:", len(PROMO_DAYS), "/", N_DAYS)
print("Hint: Stop 4 case C for THIS desk is usually hybrid.")


### 7b — Who releases what + because (TRY THIS)

This is the **accountability** half of go / no-go.

| Field | Ask out loud |
| --- | --- |
| `releases_daily_order` | If tomorrow’s Reduce/Maintain/Order is wrong, **whose system released it?** |
| `releases_email_override` | If a promo override spends money, **who stamped it?** |
| `because` | Why this recommendation on **this** Electronics log (≥ 40 characters)? |

Remember: **proposal ≠ release**. The LLM may draft; a person (or RL policy) releases.


In [ ]:
# STOP 7b — TRY THIS

releases_daily_order = ""     # <<< FILL: who releases the daily order?
releases_email_override = ""  # <<< FILL: who stamps the promo override?
because_text = ""             # <<< FILL: ≥ 40 chars; mention Electronics or inventory_dataset_full

print("Order release:", releases_daily_order or "(FILL)")
print("Override release:", releases_email_override or "(FILL)")
print("Because:", because_text or "(FILL)")


### 7c — Define `reward_ok` (RUN WITH ME)

Tiny helper: does the reward sentence use CFO language?


In [ ]:
# STOP 7c-1 — RUN WITH ME · reward_ok

def reward_ok(text: str) -> bool:
    # CFO language check: margin OR sales OR profit
    t = (text or "").lower()
    return "margin" in t or "sales" in t or "profit" in t


### 7c — Define `review_memo` (RUN WITH ME)

`review_memo` **is** the go / no-go grader for this course.

Read each check — that is what “PASS” means:
- LLM side complete (goal, tools, human_handoff)
- MDP side complete (states, actions, reward language, why_repeatable)
- recommendation is one of the four legal words
- family fit (inventory state + `LLM`-only → usually wrong → hybrid)
- `because` long enough and tied to Electronics / dataset
- both release owners named

**NOT YET** is the memo telling you it is not shippable yet. Fix gaps; re-run.


In [ ]:
# STOP 7c-2 — RUN WITH ME · review_memo

def memo_checklist(memo: dict) -> list:
    # Each item: (label, ok_bool) — used for the go/no-go board
    llm = memo.get("llm_design", {})
    mdp = memo.get("mdp_design", {})
    rec = (memo.get("recommendation") or "").strip()
    because = (memo.get("because") or "").strip()
    who_order = (memo.get("releases_daily_order") or "").strip()
    who_email = (memo.get("releases_email_override") or "").strip()

    llm_ok = bool((llm.get("goal") or "").strip() and llm.get("tools") and (llm.get("human_handoff") or "").strip())
    mdp_ok = bool(
        (mdp.get("states") or "").strip()
        and mdp.get("actions")
        and reward_ok(mdp.get("reward", ""))
        and (mdp.get("why_repeatable") or "").strip()
    )
    rec_ok = rec in {"LLM", "RL", "hybrid", "do not automate"}
    hybrid_ok = not (rec == "LLM" and "inventory" in (mdp.get("states") or "").lower())
    because_ok = len(because) >= 40 and (
        "electronics" in because.lower() or "inventory_dataset" in because.lower()
    )
    release_ok = len(who_order) >= 3 and len(who_email) >= 3

    return [
        ("LLM design", llm_ok),
        ("MDP design", mdp_ok),
        ("Recommendation", rec_ok),
        ("Family fit", hybrid_ok),
        ("Because", because_ok),
        ("Release owners", release_ok),
    ]


def review_memo(memo: dict) -> bool:
    # Return True only if the go/no-go memo is complete enough
    checks = memo_checklist(memo)
    issues = [label for label, ok in checks if not ok]
    # Friendly detail lines (same spirit as before)
    detail = []
    llm = memo.get("llm_design", {})
    mdp = memo.get("mdp_design", {})
    rec = (memo.get("recommendation") or "").strip()
    because = (memo.get("because") or "").strip()
    who_order = (memo.get("releases_daily_order") or "").strip()
    who_email = (memo.get("releases_email_override") or "").strip()
    if not (llm.get("goal") or "").strip() or not llm.get("tools") or not (llm.get("human_handoff") or "").strip():
        detail.append("LLM needs goal, tools, human_handoff")
    if not (mdp.get("states") or "").strip() or not mdp.get("actions") or not reward_ok(mdp.get("reward", "")):
        detail.append("MDP needs states, actions, reward with margin/sales/profit")
    if not (mdp.get("why_repeatable") or "").strip():
        detail.append("MDP needs why_repeatable")
    if rec not in {"LLM", "RL", "hybrid", "do not automate"}:
        detail.append("recommendation must be LLM / RL / hybrid / do not automate")
    if rec == "LLM" and "inventory" in (mdp.get("states") or "").lower():
        detail.append("inventory state + LLM-only → usually hybrid")
    if len(because) < 40:
        detail.append("because too short")
    if "electronics" not in because.lower() and "inventory_dataset" not in because.lower():
        detail.append("because must mention Electronics or inventory_dataset_full")
    if len(who_order) < 3 or len(who_email) < 3:
        detail.append("name both release owners")

    if detail:
        print("NOT YET")
        for issue in detail:
            print(" -", issue)
        return False

    print("PASS")  # keep check short; discuss aloud from MEMO fields
    return True


### 7c — Assemble `MEMO`, validate, and show the go / no-go board (RUN / TRY)

Wire Stop 5–7b into one `MEMO` dict. Then:
1. `review_memo(MEMO)` → **PASS** or **NOT YET** (the go / no-go gate)
2. checklist board → which sections are complete
3. release lanes → MDP/RL path · LLM path · human stamp

If anything is empty, jump back to the FILL and re-run forward.  
**Class win condition:** `review_memo` prints **PASS** and both partners can explain the board.


In [ ]:
# STOP 7c-3 — assemble MEMO + review + go/no-go board

MEMO = {
    "process": "Electronics replenishment exceptions (inventory_dataset_full.csv)",
    "llm_design": {
        "goal": llm_goal,
        "tools": list(llm_tools),
        "knowledge": llm_knowledge,
        "human_handoff": llm_human_handoff,
    },
    "mdp_design": {
        "states": mdp_states,
        "actions": list(mdp_actions),
        "reward": mdp_reward,
        "why_repeatable": mdp_why_repeatable,
    },
    "recommendation": (say_it or "").strip(),
    "releases_daily_order": releases_daily_order,
    "releases_email_override": releases_email_override,
    "because": because_text,
}

ok = review_memo(MEMO)
checks = memo_checklist(MEMO)
print("board:", {label: ("ok" if passed else "fix") for label, passed in checks})

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

    labels = [c[0] for c in checks]
    vals = [1 if c[1] else 0 for c in checks]
    colors = ["#2F5D50" if v else "#8B3A2A" for v in vals]
    axes[0].barh(labels[::-1], vals[::-1], color=colors[::-1])
    axes[0].set_xlim(0, 1.2)
    axes[0].set_xticks([0, 1], ["fail", "ok"])
    axes[0].set_title("Go / no-go checklist")

    axes[1].axis("off")
    axes[1].set_xlim(0, 10)
    axes[1].set_ylim(0, 6)
    axes[1].set_title("Who releases what")
    lanes = [
        (0.4, 3.6, 4.2, 1.6, "MDP / RL lane", "daily Order/Maintain/Reduce\nfrom Current_Inventory", "#E8F0EC"),
        (5.2, 3.6, 4.4, 1.6, "LLM lane", "promo/supplier brief\n→ draft override only", "#E8EEF5"),
        (2.5, 1.0, 5.0, 1.6, "Human stamp", "desk lead releases override\n(proposal ≠ release)", "#F7EDE6"),
    ]
    for x, y, w, h, title, body, color in lanes:
        rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="#333333", lw=1.1)
        axes[1].add_patch(rect)
        axes[1].text(x + w / 2, y + h - 0.35, title, ha="center", fontsize=9, fontweight="bold")
        axes[1].text(x + w / 2, y + 0.45, body, ha="center", fontsize=7)
    rec = (MEMO.get("recommendation") or "?").strip() or "?"
    axes[1].text(
        5,
        0.35,
        f"class vote: {rec}   ·   review_memo: {'PASS' if ok else 'NOT YET'}",
        ha="center",
        fontsize=9,
    )
    fig.tight_layout()
    plt.show()


## Stop 8 — LAB (10–15 min) on real rows

![Keep the loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/agent-loop.png)


### Challenge 1 — Mean profit: promo vs normal


In [ ]:
# CH1 — TRY THIS

promo_profit = None   # <<< FILL: mean Profit where Is_Promotional is True
normal_profit = None  # <<< FILL: mean Profit where Is_Promotional is False

# Hint:
# promo_profit = sum(r["Profit"] for r in ROWS if r["Is_Promotional"]) / len(PROMO_DAYS)
# normal_profit = sum(r["Profit"] for r in ROWS if not r["Is_Promotional"]) / (N_DAYS - len(PROMO_DAYS))

print(promo_profit, normal_profit)
if promo_profit is None or normal_profit is None:
    print("NOT YET — compute from ROWS")
else:
    print("CH1 recorded. Discuss: does promo change money enough to need an override path?")
    if plt is not None:
        fig, ax = plt.subplots(figsize=(4.2, 2.8))
        ax.bar(["promo", "normal"], [promo_profit, normal_profit], color=["#C45C26", "#2F5D50"])
        ax.set_ylabel("mean Profit")
        ax.set_title("Validation: promo vs normal money")
        fig.tight_layout()
        plt.show()


### Challenge 1b — CHANGE ME · filter the real log

**PAIR 20s.** Change `ONLY_PROMO`. Re-run mean profit.

**Discuss:** Do promo days look more profitable here? Does that alone justify auto-ordering?


In [ ]:
# CH1b — CHANGE ME · money filter

ONLY_PROMO = True  # <<< CHANGE ME: True=promo days, False=non-promo

subset = [r for r in ROWS if r["Is_Promotional"] is ONLY_PROMO]
mean_p = statistics.mean(r["Profit"] for r in subset) if subset else float("nan")
print("filter ONLY_PROMO=", ONLY_PROMO, "n=", len(subset), "mean Profit=", round(mean_p, 1))

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.hist([r["Profit"] for r in subset], bins=20, color="#1F4E79", edgecolor="white")
    ax.set_title("Profit distribution (" + ("promo" if ONLY_PROMO else "normal") + ")")
    ax.set_xlabel("Profit")
    fig.tight_layout()
    plt.show()


### Challenge 2 — Worst stockout day in the real log


In [ ]:
# CH2 — TRY THIS

day = max(ROWS, key=lambda r: r["Stockout_Units"])  # real worst stockout day
print("Worst stockout:", day["Date"], day["Stockout_Units"], "promo=", day["Is_Promotional"])

family_for_recurring_stockout_plus_notes = ""  # <<< FILL if ops notes accompany stockouts
print("CHECK:", "PASS" if family_for_recurring_stockout_plus_notes == "hybrid" else "NOT YET — usually hybrid")


### Challenge 3 — Break the memo on purpose, then fix (RUN WITH ME)

We force a **bad GO** (`recommendation = LLM` on an inventory MDP) so you see why `review_memo` exists.  
Then we restore the real memo — that is the difference between a vibes vote and a go / no-go gate.


In [ ]:
# CH3 — RUN WITH ME · break then fix (validate the validator)

bad = dict(MEMO)
bad["recommendation"] = "LLM"  # wrong family on purpose
print("broken:"); broken_ok = review_memo(bad)
print("fixed:"); fixed_ok = review_memo(MEMO)

if plt is not None:
    bad_checks = memo_checklist(bad)
    good_checks = memo_checklist(MEMO)
    labels = [c[0] for c in good_checks]
    fig, ax = plt.subplots(figsize=(6.5, 3.0))
    y = range(len(labels))
    ax.barh([i + 0.15 for i in y], [1 if c[1] else 0 for c in bad_checks], height=0.3, color="#8B3A2A", label="broken")
    ax.barh([i - 0.15 for i in y], [1 if c[1] else 0 for c in good_checks], height=0.3, color="#2F5D50", label="fixed")
    ax.set_yticks(list(y), labels)
    ax.set_xlim(0, 1.2)
    ax.set_xticks([0, 1], ["fail", "ok"])
    ax.set_title("Validation: review_memo catches a bad go/no-go")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()


## Stop 9 — Framework line (keep forever)

One sentence for **your** project: observe / decide / reward / handoff.


In [ ]:
# STOP 9 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL: one keepable sentence

print("Family:", framework_family or "(FILL)")
print("I keep:", framework_line or "(FILL)")


## Raise a hand — real case complete

1. [ ] Partner can explain **GO vs NO-GO** in one sentence each  
2. [ ] Desk dashboard shows inventory + promo markers  
3. [ ] MDP card validation **PASS** (Stop 5b)  
4. [ ] Policy-vs-history plot discussed (Stop 2d)  
5. [ ] `review_memo` prints **PASS** (go / no-go gate)  
6. [ ] Go / no-go board shows MDP lane + LLM lane + human stamp  
7. [ ] Partner can name **both** release owners  
